# Unidade II — Análise de Dados

## Tipos e descrição de dados

**Carga estimada:** 3 horas e 30 minutos  
**Pré-requisitos:** Python básico, média e operações algébricas.

> **Pergunta norteadora:** como o tipo de um atributo determina quais operações, resumos e interpretações são válidos?


## Objetivos de aprendizagem

Ao concluir este notebook, você será capaz de:

- distinguir objetos, atributos e matrizes de dados;
- classificar atributos como nominais, binários, ordinais ou numéricos;
- diferenciar atributos discretos e contínuos;
- calcular e interpretar medidas de tendência central e dispersão;
- reconhecer o efeito de valores extremos e da escala sobre os resumos.


## Objetos e atributos

Um **objeto de dados** é a entidade observada: cliente, flor, transação ou instante. Um **atributo** descreve uma característica do objeto. Em uma tabela usual, linhas representam objetos e colunas representam atributos. Essa convenção não é universal — séries temporais, imagens, textos e grafos exigem outras estruturas.

| Tipo | Característica | Operações com significado | Exemplo |
|---|---|---|---|
| Nominal | categorias sem ordem | igualdade e contagem | município |
| Binário | dois estados | igualdade; presença/ausência | inadimplente |
| Ordinal | categorias ordenadas | ordem e comparação | satisfação baixa/média/alta |
| Intervalar | diferenças comparáveis; zero convencional | soma e diferença | temperatura em °C |
| Razão | diferenças e razões; zero significativo | operações aritméticas | massa em kg |

Um código numérico não torna um atributo quantitativo. Se `1 = norte`, `2 = sul` e `3 = leste`, calcular a média dos códigos não produz uma região significativa.


### Discreto e contínuo

Atributos **discretos** assumem valores de um conjunto finito ou enumerável, como número de chamados. Atributos **contínuos** são conceitualmente medidos em uma escala contínua, como duração. A representação digital sempre tem precisão finita, mas isso não muda necessariamente a natureza da grandeza.

> **U02-NB01-V01 — Verifique seu entendimento:** CEP, ano civil, avaliação de uma a cinco estrelas e renda são numéricos no arquivo. Classifique-os pela natureza e indique uma operação que seria inadequada para cada um.


## Um conjunto misto

A tabela sintética abaixo combina tipos distintos e um valor extremo proposital. Ela será pequena para permitir a conferência manual.


In [1]:
import numpy as np
import pandas as pd

dados = pd.DataFrame({
    "cliente": ["A", "B", "C", "D", "E", "F"],
    "plano": ["básico", "básico", "premium", "premium", "básico", "premium"],
    "satisfacao": pd.Categorical(
        ["baixa", "média", "alta", "média", "baixa", "alta"],
        categories=["baixa", "média", "alta"], ordered=True,
    ),
    "chamados": [2, 1, 0, 3, 2, 10],
    "mensalidade": [80, 85, 120, 125, 90, 500],
    "renovou": [True, True, True, False, False, True],
})
dados


,cliente,plano,satisfacao,chamados,mensalidade,renovou
0,A,básico,baixa,2,80,True
1,B,básico,média,1,85,True
2,C,premium,alta,0,120,True
3,D,premium,média,3,125,False
4,E,básico,baixa,2,90,False
5,F,premium,alta,10,500,True


## Tendência central

Para valores $x_1,\ldots,x_n$, a média aritmética é

$$
\bar{x}=\frac{1}{n}\sum_{i=1}^{n}x_i.
$$

A **mediana** é o valor central após ordenar os dados; com quantidade par, usa-se a média dos dois centrais. A **moda** é o valor mais frequente e também se aplica a atributos nominais. A média usa magnitudes e é sensível a extremos; a mediana depende da ordem e é mais robusta.


In [2]:
mensalidades = dados["mensalidade"].to_numpy()
media_manual = mensalidades.sum() / mensalidades.size
ordenadas = np.sort(mensalidades)
mediana_manual = (ordenadas[2] + ordenadas[3]) / 2

pd.Series({
    "média manual": media_manual,
    "média pandas": dados["mensalidade"].mean(),
    "mediana manual": mediana_manual,
    "mediana pandas": dados["mensalidade"].median(),
    "moda do plano": dados["plano"].mode().iat[0],
}, name="resultado")


média manual      166.666667
média pandas      166.666667
mediana manual         105.0
mediana pandas         105.0
moda do plano         básico
Name: resultado, dtype: object

O valor 500 desloca a média para cima, enquanto a mediana permanece entre os valores centrais 90 e 120. Nenhuma medida é universalmente melhor: a escolha depende da distribuição e da pergunta.

## Dispersão

A amplitude é $\max(x)-\min(x)$. A amplitude interquartil é $IQR=Q_3-Q_1$. Para uma amostra, a variância e o desvio-padrão são

$$
s^2=\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2,
\qquad s=\sqrt{s^2}.
$$

O denominador $n-1$ é usado na variância amostral como correção para estimar a variância populacional. A variância fica em unidades ao quadrado; o desvio-padrão retorna à unidade original.


In [3]:
desvios = mensalidades - media_manual
variancia_manual = np.sum(desvios**2) / (mensalidades.size - 1)
q1, q3 = dados["mensalidade"].quantile([0.25, 0.75])

pd.Series({
    "amplitude": mensalidades.max() - mensalidades.min(),
    "IQR": q3 - q1,
    "variância manual": variancia_manual,
    "variância pandas": dados["mensalidade"].var(ddof=1),
    "desvio-padrão": dados["mensalidade"].std(ddof=1),
}).round(2)


amplitude             420.00
IQR                    37.50
variância manual    27016.67
variância pandas    27016.67
desvio-padrão         164.37
dtype: float64

## Covariância e correlação

A covariância amostral entre $X$ e $Y$ é

$$
s_{XY}=\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y}).
$$

Seu sinal indica direção linear, mas sua magnitude depende das unidades. A correlação de Pearson padroniza a covariância:

$$
r_{XY}=\frac{s_{XY}}{s_Xs_Y}, \qquad -1\le r_{XY}\le 1.
$$

Correlação próxima de zero significa ausência de relação **linear**, não independência; correlação também não demonstra causalidade.


In [4]:
numericos = dados[["chamados", "mensalidade"]]
print("Covariância amostral:")
display(numericos.cov().round(2))
print("Correlação de Pearson:")
display(numericos.corr().round(3))


Covariância amostral:


,chamados,mensalidade
chamados,12.8,560.00
mensalidade,560.0,27016.67


Correlação de Pearson:


,chamados,mensalidade
chamados,1.000,0.952
mensalidade,0.952,1.000


Com apenas seis observações e um caso extremo, a correlação é instável e não deve receber interpretação causal. Este exemplo mostra por que um coeficiente precisa ser acompanhado por contexto e visualização.

> **U02-NB01-V02 — Verifique seu entendimento:** se todos os valores de mensalidade fossem multiplicados por 100, quais medidas mudariam? Compare média, mediana, variância, desvio-padrão e correlação.

> **U02-NB01-E01 — Exercício:** remova temporariamente a mensalidade 500, recalcule os resumos e escreva um parágrafo comparando a sensibilidade de média, mediana, amplitude, IQR e desvio-padrão. Não conclua que o caso deve ser removido: discuta quais informações seriam necessárias para decidir.


## Síntese

- A natureza do atributo determina operações e resumos válidos.
- Códigos numéricos podem representar categorias, não quantidades.
- Média e desvio-padrão são sensíveis a extremos; mediana e IQR são mais robustos.
- Covariância depende da escala; correlação mede associação linear padronizada.
- Resumos devem ser interpretados com distribuição, contexto e tamanho da amostra.

## Referências

- HAN, Jiawei; PEI, Jian; TONG, Hanghang. *Data Mining: Concepts and Techniques*. 4. ed. Cambridge: Morgan Kaufmann/Elsevier, 2023. Cap. 2, seções 2.1 e 2.2.
